# 01 — Radar bed-pick coverage over lake 1951 (2019cw428)

Step 0 of the MGSIM bed-ensemble experiment: what ice-penetrating-radar bed
picks exist inside a **50 km × 50 km AOI oriented like the simulation domain**
(along-flow frame, concentric with the Elmer domain),
and how do they sit relative to the **Elmer simulation domain** (24.75 km) and
the site's **analysis tile** (14.75 km)?

Data: CReSIS L2 `csv_good` picks at full posting density (~15 m along-track),
pulled by `../scripts/fetch_radar_lines.py` (browse aggregates discover the
crossing segments; only those segments are fetched in full). Re-run that
script first if the derived files are missing. Basemap: the published viscous
nc (`cinematic/source_data/pnas_sims/`, populated by `cinematic/fetch_sources.sh`).

Frames: `X,Y` are EPSG:3413 meters; `LX,LY` are along-flow local meters with
origin at the domain center (+x = ice-flow direction). Elmer DEM coordinates
are `LX + 12375, LY + 12375`.

Vertical datum: CReSIS `SURFACE`/`BOTTOM` are **depths below the platform**
(`BOTTOM − SURFACE = THICK`), so bed elevation is `BED_ELEV = ELEVATION −
BOTTOM` (WGS-84 ellipsoidal — expect ≈ +28 m vs the geoid-referenced
published bed). The fetch script QCs each season's median residual against
the published bed; seasons whose ELEVATION reference is something else
entirely get `DATUM_SUSPECT=True` (the ICORDS/ACORDS-era 2003 and 2005
campaigns here). Their `THICK` is still good — the conditioning step can
rebuild their bed as `surface DEM − THICK`, which never touches the aircraft
datum.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DERIVED = Path('../data/derived')
FIGS = Path('../figures'); FIGS.mkdir(exist_ok=True)

picks = pd.read_csv(DERIVED / 'radar_picks_cw1951_aoi50km.csv')
aoi = json.load(open(DERIVED / 'aoi_cw1951.json'))
summary = json.load(open(DERIVED / 'fetch_summary.json'))

tbl = pd.DataFrame(summary).T.sort_index()
tbl['dates'] = tbl['dates'].apply(lambda d: ', '.join(d[:3]) + ('…' if len(d) > 3 else ''))
print(f"{len(picks)} picks in AOI · {picks.IN_SIMDOMAIN.sum()} in Elmer domain · "
      f"{picks.IN_TILE.sum()} in analysis tile · {picks.SEASON.nunique()} seasons · "
      f"{picks.DATUM_SUSPECT.sum()} picks in datum-suspect seasons")
tbl[['n_aoi', 'n_simdomain', 'n_tile', 'track_km_aoi', 'n_frames',
     'bed_qc_median_m', 'datum_suspect', 'dates']]

In [ ]:
# Domain geometry in both frames, plus the center-lake outline.
center = np.array(aoi['domain_center_3413'])
ex, ey = np.array(aoi['ex_3413']), np.array(aoi['ey_3413'])

def quad_3413(half_m):
    c = np.array([[-1, -1], [1, -1], [1, 1], [-1, 1], [-1, -1]]) * half_m
    return center + np.outer(c[:, 0], ex) + np.outer(c[:, 1], ey)

def quad_local(half_m):
    return np.array([[-1, -1], [1, -1], [1, 1], [-1, 1], [-1, -1]]) * half_m

TILE, ELMER = aoi['tile_half_m'], aoi['elmer_half_m']
aoi_quad = np.array(aoi['aoi_corners_3413'] + [aoi['aoi_corners_3413'][0]])

# Center-lake ring: rings_uv are in [0,1]^2 over the 14.75 km tile; the v-axis
# convention (image-down vs math-up) is auto-resolved against the known
# lake-centroid position in the local frame.
simlakes = json.load(open('../../cinematic/scene_ch02/output_simdomains/simlakes_CW2019_1951.json'))
clake = next(l for l in simlakes['lakes'] if l.get('center'))
cll = np.array(aoi['centroid_3413']) - center
c_true = np.array([cll @ ex, cll @ ey])
uv = np.array(clake['rings_uv'][0])
cand = [np.stack([(uv[:, 0] - 0.5) * 2 * TILE, s * (uv[:, 1] - 0.5) * 2 * TILE], 1)
        for s in (+1, -1)]
ring_local = min(cand, key=lambda r: np.hypot(*(r.mean(0) - c_true)))
ring_3413 = center + np.outer(ring_local[:, 0], ex) + np.outer(ring_local[:, 1], ey)
print('lake ring vertices:', len(ring_local),
      '· ring-centroid offset from true centroid:',
      f"{np.hypot(*(ring_local.mean(0) - c_true)):.0f} m")

In [ ]:
# Panel A — coverage in EPSG:3413. One color per season with data.
seasons = sorted(picks.SEASON.unique())
cmap = plt.get_cmap('tab20')
fig, ax = plt.subplots(figsize=(9.5, 9.5))
for i, s in enumerate(seasons):
    g = picks[picks.SEASON == s]
    ax.scatter(g.X / 1e3, g.Y / 1e3, s=2.5, color=cmap(i % 20),
               label=f"{s} ({len(g)})", rasterized=True)
for half, lw, ls, lab in [(ELMER, 1.8, '-', 'Elmer domain 24.75 km'),
                          (TILE, 1.2, '--', 'analysis tile 14.75 km')]:
    q = quad_3413(half) / 1e3
    ax.plot(q[:, 0], q[:, 1], ls, color='k', lw=lw, label=lab)
ax.plot(ring_3413[:, 0] / 1e3, ring_3413[:, 1] / 1e3, color='#b8860b', lw=1.6,
        label='lake 1951 outline')
ax.plot(aoi_quad[:, 0] / 1e3, aoi_quad[:, 1] / 1e3, color='0.4', lw=1.0,
        label='50 km AOI (along-flow oriented)')
ax.set_aspect('equal'); ax.set_xlabel('EPSG:3413 x [km]'); ax.set_ylabel('y [km]')
ax.set_title('CReSIS bed picks — lake 1951 AOI')
ax.legend(fontsize=7.5, loc='upper left', ncols=2, framealpha=0.9)
fig.tight_layout(); fig.savefig(FIGS / 'coverage_3413.png', dpi=160)
plt.show()

In [ ]:
# Panel B — local along-flow frame over the published bed (viscous nc).
# Datum-good picks share the bed colormap via BED_ELEV; the two
# datum-suspect seasons (2003/2005) are drawn in light gray.
import xarray as xr
NC = Path('../../cinematic/source_data/pnas_sims/2019cw428_d25km_r2500m_csi6.5e6_tsic_viscous.nc')
fig, ax = plt.subplots(figsize=(10.5, 9.5))
bed_grid = None
if NC.exists():
    ds = xr.open_dataset(NC)
    bed = ds.topo_bed
    for d in ('case', 'level'):
        if d in bed.dims: bed = bed.isel({d: 0})
    xs, ys = ds.x.values, ds.y.values
    arr = bed.values
    if ys[0] > ys[-1]: ys, arr = ys[::-1], arr[::-1]
    vmin, vmax = float(np.nanmin(arr)), float(np.nanmax(arr))
    pm = ax.pcolormesh(xs / 1e3, ys / 1e3, arr, cmap='gist_earth',
                       vmin=vmin, vmax=vmax, shading='auto')
    plt.colorbar(pm, ax=ax, shrink=0.75, label='bed elevation [m] (published grid + radar dots)')
    bed_grid = (xs, ys, arr)
else:
    vmin = vmax = None
    ax.text(0, 0, 'nc basemap missing —\nrun cinematic/fetch_sources.sh', ha='center')
bad = picks[picks.DATUM_SUSPECT]
good = picks[~picks.DATUM_SUSPECT]
ax.scatter(bad.LX / 1e3, bad.LY / 1e3, s=2.5, color='0.82', rasterized=True,
           label=f'datum-suspect seasons ({bad.SEASON.nunique()})')
# ~28 m ellipsoid-to-geoid shift so the dot colors are comparable to the grid
ax.scatter(good.LX / 1e3, good.LY / 1e3, c=good.BED_ELEV - 28.0, s=4,
           cmap='gist_earth', vmin=vmin, vmax=vmax,
           edgecolors='k', linewidths=0.1, rasterized=True,
           label=f'bed picks, datum-good ({good.SEASON.nunique()} seasons)')
for half, lw, ls in [(ELMER, 1.8, '-'), (TILE, 1.2, '--')]:
    q = quad_local(half) / 1e3
    ax.plot(q[:, 0], q[:, 1], ls, color='k', lw=lw)
ax.plot(ring_local[:, 0] / 1e3, ring_local[:, 1] / 1e3, color='#b8860b', lw=1.6)
ax.annotate('ice flow', xy=(10, -13.5), xytext=(2, -13.5), fontsize=10,
            arrowprops=dict(arrowstyle='->', lw=1.4))
lim = aoi['aoi_half_m'] / 1e3
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect('equal')
ax.set_xlabel('along-flow LX [km]'); ax.set_ylabel('across-flow LY [km]')
ax.set_title('bed picks in the along-flow domain frame')
ax.legend(fontsize=8.5, loc='upper left', framealpha=0.9)
fig.tight_layout(); fig.savefig(FIGS / 'coverage_local.png', dpi=160)
plt.show()

In [ ]:
# Radar bed vs published bed at the pick locations (analysis tile,
# datum-good seasons). The systematic part of the offset is the geoid
# undulation (~+28 m: radar is WGS-84 ellipsoidal, BedMachine is
# geoid-referenced); the spread around it is the along-line signal MGSIM
# will condition on.
if bed_grid is not None:
    from scipy.interpolate import RegularGridInterpolator
    xs, ys, arr = bed_grid
    itp = RegularGridInterpolator((ys, xs), arr, bounds_error=False)
    t = picks[picks.IN_TILE & ~picks.DATUM_SUSPECT].copy()
    t['BED_PUB'] = itp(np.stack([t.LY, t.LX], 1))
    t = t.dropna(subset=['BED_PUB'])
    dz = t.BED_ELEV - t.BED_PUB
    fig, axs = plt.subplots(1, 3, figsize=(15.5, 4.6))
    axs[0].scatter(t.BED_PUB, t.BED_ELEV, s=4, alpha=0.35, rasterized=True)
    lo, hi = np.percentile(np.r_[t.BED_PUB, t.BED_ELEV], [0.2, 99.8])
    axs[0].plot([lo, hi], [lo, hi], 'k--', lw=1, label='1:1')
    axs[0].plot([lo, hi], [lo + 28, hi + 28], 'r--', lw=1, label='1:1 + 28 m geoid')
    axs[0].legend(fontsize=8)
    axs[0].set_xlabel('published bed [m, geoid]')
    axs[0].set_ylabel('radar BED_ELEV [m, ellipsoid]')
    axs[1].hist(dz, bins=60, color='#888', range=np.percentile(dz, [0.2, 99.8]))
    axs[1].axvline(28, color='r', lw=1.2, ls='--')
    axs[1].axvline(dz.median(), color='k', lw=1.2)
    axs[1].set_xlabel('radar − published [m]')
    axs[1].set_title(f"n={len(t)}  median {dz.median():+.1f} m  std {dz.std():.1f} m")
    per = dz.groupby(t.SEASON).median().sort_values()
    axs[2].barh(range(len(per)), per.values, color='#6b8dad')
    axs[2].set_yticks(range(len(per)), [s.replace('_Greenland', '') for s in per.index],
                      fontsize=7.5)
    axs[2].axvline(28, color='r', lw=1.2, ls='--')
    axs[2].set_xlabel('per-season median offset [m]')
    fig.suptitle('bed picks vs published bed inside the analysis tile (datum-good seasons)',
                 y=1.02)
    fig.tight_layout(); fig.savefig(FIGS / 'radar_vs_published_bed.png', dpi=160,
                                    bbox_inches='tight')
    plt.show()
    print('Spread beyond the 28 m datum line = disagreement between the radar bed\n'
          'and the BedMachine-based published bed — the uncertainty MGSIM will model.')
else:
    print('skipped — nc basemap not available')

## Next

- **02 — variograms & detrending**: fix the ellipsoid→geoid datum, rebuild the
  2003/2005 beds as `surface DEM − THICK`, subtract the published
  (BedMachine-based) bed as the trend, fit the residual variogram with
  `multigrid-sgsim` / GStatSim tooling.
- **03 — bed realizations**: N≈10 seeded conditional MGSIM realizations on the
  166×166 (150 m) Elmer DEM grid, tapered to the published bed over the outer
  ~2 km so the wall BCs stay identical; write `.dat` DEMs in the sif format.
- **04 — Elmer ensemble**: swap each bed into the published viscous setup
  (everything else held fixed) → N surface-stress fields → ensemble spread.